In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [2]:
import os
import json
import pickle
import pandas as pd
import torch
import torchaudio
import re
import torch.nn as nn
import torchaudio.transforms as T
import soundfile as sf
from datetime import datetime
from tqdm.auto import tqdm
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from concurrent.futures import ProcessPoolExecutor

2024-11-24 16:18:29.972034: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1732432710.002924    6006 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1732432710.012201    6006 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-24 16:18:30.043008: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 새로운 구성에 맞게 다시 작성하기

In [3]:
preprocessed_train_audio_path = "/home/wangan/Data/Training/preprocessed_audio_train/"
preprocessed_test_audio_path = "/home/wangan/Data/Validation/preprocessed_audio_test/"
pr_train=os.listdir(preprocessed_train_audio_path)
pr_test=os.listdir(preprocessed_test_audio_path)

pr_train_path = [os.path.join(preprocessed_train_audio_path, p) for p in pr_train]
pr_test_path = [os.path.join(preprocessed_test_audio_path, p) for p in pr_test]

## 훈련데이터 경로 데이터프레임

In [4]:
# 각 파일 경로에서 ID 추출
ids = [re.search(r'/([\d]{4})-', path).group(1) for path in pr_train_path]

# DataFrame 생성
df = pd.DataFrame({'file_path': pr_train_path, 'id': ids})

# ID와 파일명의 숫자 부분을 추출하는 함수
def extract_sort_keys(file_path):
    id_match = re.search(r'/([\d]{4})-', file_path)
    file_num_match = re.search(r'-(\d+)\.mp3', file_path)
    id_key = id_match.group(1) if id_match else '0000'
    file_num_key = int(file_num_match.group(1)) if file_num_match else 0
    return id_key, file_num_key

# 정렬 키 추출 및 정렬
df['sort_keys'] = df['file_path'].apply(extract_sort_keys)
df_sorted = df.sort_values(by='sort_keys')

# 정렬 키 열 제거 및 인덱스 재설정
df_sorted = df_sorted.drop('sort_keys', axis=1).reset_index(drop=True)

# 결과 출력
df_sorted

,file_path,id
0,/home/wangan/Data/Training/preprocessed_audio_...,0002
1,/home/wangan/Data/Training/preprocessed_audio_...,0002
2,/home/wangan/Data/Training/preprocessed_audio_...,0002
3,/home/wangan/Data/Training/preprocessed_audio_...,0002
4,/home/wangan/Data/Training/preprocessed_audio_...,0002
...,...,...
20127,/home/wangan/Data/Training/preprocessed_audio_...,5250
20128,/home/wangan/Data/Training/preprocessed_audio_...,5250
20129,/home/wangan/Data/Training/preprocessed_audio_...,5250
20130,/home/wangan/Data/Training/preprocessed_audio_...,5250


## 테스트데이터 경로 데이터프레임

In [5]:
# 각 파일 경로에서 ID 추출
ids = [re.search(r'/([\d]{4})-', path).group(1) for path in pr_test_path]

# DataFrame 생성
df = pd.DataFrame({'file_path': pr_test_path, 'id': ids})

# ID와 파일명의 숫자 부분을 추출하는 함수
def extract_sort_keys(file_path):
    id_match = re.search(r'/([\d]{4})-', file_path)
    file_num_match = re.search(r'-(\d+)\.mp3', file_path)
    id_key = id_match.group(1) if id_match else '0000'
    file_num_key = int(file_num_match.group(1)) if file_num_match else 0
    return id_key, file_num_key

# 정렬 키 추출 및 정렬
df['sort_keys'] = df['file_path'].apply(extract_sort_keys)
df_sorted_test = df.sort_values(by='sort_keys')

# 정렬 키 열 제거 및 인덱스 재설정
df_sorted_test = df_sorted_test.drop('sort_keys', axis=1).reset_index(drop=True)

# 결과 출력
df_sorted_test

,file_path,id
0,/home/wangan/Data/Validation/preprocessed_audi...,0008
1,/home/wangan/Data/Validation/preprocessed_audi...,0008
2,/home/wangan/Data/Validation/preprocessed_audi...,0008
3,/home/wangan/Data/Validation/preprocessed_audi...,0008
4,/home/wangan/Data/Validation/preprocessed_audi...,0008
...,...,...
2515,/home/wangan/Data/Validation/preprocessed_audi...,5146
2516,/home/wangan/Data/Validation/preprocessed_audi...,5146
2517,/home/wangan/Data/Validation/preprocessed_audi...,5146
2518,/home/wangan/Data/Validation/preprocessed_audi...,5146


In [ ]:
def generate_audio_embeddings(audio_paths, model_name):
    # Wav2Vec2 모델과 프로세서 로드
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name)
    
    # GPU 사용 가능 시 모델을 GPU로 이동
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = 'cuda:0'
    model = model.to(device)

    embeddings = []
    
    for audio_path in tqdm(audio_paths, desc="Generating embeddings"):
        # 오디오 파일 로드 및 전처리
        waveform, orig_sr = torchaudio.load(audio_path)
        
        # 16 kHz로 리샘플링
        sample_rate = 16000
        if orig_sr != sample_rate:
            resampler = torchaudio.transforms.Resample(orig_sr, sample_rate)
            waveform = resampler(waveform)

        # 스테레오를 모노로 변환 (필요한 경우)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # 텐서를 리스트로 변환
        waveform = waveform.squeeze().tolist()
        
        # Wav2Vec2Processor 적용
        inputs = processor(waveform, sampling_rate=sample_rate, return_tensors="pt", padding=True)
        
        # 입력을 GPU로 이동 (가능한 경우)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 모델에 입력 전달
        with torch.no_grad():
            outputs = model(**inputs)
        
        # 시퀀스 차원에 대해 평균을 취해 고정 크기 벡터 생성
        embedding = torch.mean(outputs.last_hidden_state, dim=1)
        
        embeddings.append(embedding.cpu())  # CPU로 다시 이동
    
    # 모든 임베딩을 하나의 텐서로 결합
    all_embeddings = torch.cat(embeddings, dim=0)
    
    return all_embeddings



### 훈련 오디오 임베딩

In [ ]:
# 사용 예시
audio_paths = df_sorted['file_path'].to_list()  # 오디오 파일 경로 리스트
model_name = "kresnik/wav2vec2-large-xlsr-korean"
embeddings = generate_audio_embeddings(audio_paths, model_name)

print(embeddings.shape)  # 출력: torch.Size([파일 수, 임베딩 차원])

Generating embeddings:   0%|          | 0/20132 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os

# 저장할 디렉토리 경로
save_dir = '/home/wangan/Embeddings_training/audio/wav2vec2/'

# 파일명
filename = 'audio_wav2vec_train_embeddings.pt'

# 전체 경로 생성
full_path = os.path.join(save_dir, filename)

# 임베딩 저장
torch.save(embeddings, full_path)

# # 나중에 임베딩 불러오기
# loaded_embeddings = torch.load('/home/wangan/Embeddings/audio/audio_train_embeddings.pt')

# print(loaded_embeddings.shape)  # 원본 embeddings와 동일한 shape를 가져야 합니다

### 테스트 오디오 임베딩

In [7]:
# 사용 예시
audio_paths = df_sorted_test['file_path'].to_list()  # 오디오 파일 경로 리스트
model_name = "kresnik/wav2vec2-large-xlsr-korean"
embeddings = generate_audio_embeddings(audio_paths, model_name)

print(embeddings.shape)  # 출력: torch.Size([파일 수, 임베딩 차원])


Generating embeddings:   0%|          | 0/2520 [00:00<?, ?it/s]

torch.Size([2520, 1024])


In [ ]:
import os

# 저장할 디렉토리 경로
save_dir = '/home/wangan/Embeddings_test/audio/wav2vec2/'

# 파일명
filename = 'audio_wav2vec_test_embeddings.pt'

# 전체 경로 생성
full_path = os.path.join(save_dir, filename)

# 임베딩 저장
torch.save(embeddings, full_path)

# # 나중에 임베딩 불러오기
# loaded_embeddings = torch.load('/home/wangan/Embeddings/audio/audio_test_embeddings.pt')

# print(loaded_embeddings.shape)  # 원본 embeddings와 동일한 shape를 가져야 합니다

: 